# 06 — Instruction optimization: COPRO and MIPROv2

**What you'll learn**

- The second θ slot: optimizers that rewrite each predictor's **instructions** (vs. notebook 05, which filled `demos`).
- `COPRO` — hill-climbing on instructions, in the lineage of OPRO ("LLMs as optimizers").
- `MIPROv2` — grounded instruction proposal + Bayesian search over instruction×demo configurations.
- Paper ↔ code maps for OPRO (Yang et al., 2023) and MIPROv2 (Opsahl-Ong et al., 2024).

*Time: ~30 min. Cost: ~$0.30–0.60 — the priciest notebook so far; both optimizers run many candidate evaluations. Cached reruns are free.*

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import dspy
from dotenv import load_dotenv              # pip install -r requirements.txt if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead
# STRONG_MODEL = "openrouter/moonshotai/kimi-k2.6"
# STRONG_MODEL = "openrouter/openai/gpt-4o"

lm = dspy.LM(MODEL, max_tokens=1000)        # cache=True by default: reruns are ~free
dspy.configure(lm=lm)

### Phoenix observability (optional)

Run the next cell to stream this notebook's LM calls into a local [Phoenix](https://phoenix.arize.com) trace UI — or skip it; nothing below depends on tracing. It finds or starts the Phoenix server, prints the link, and names the session by date (`dspy-lab-YYYY-MM-DD`) so different days never mix. Notebook 11 tells the full story.

In [ ]:
# optional: Phoenix tracing (see notebook 11)
import obs

obs.enable_phoenix()

### MLflow observability (optional)

Run the next cell to record this notebook's LM calls with MLflow instead (or as well): logs land in a repo-local store that is safe against remnants and reruns, and the second line finds or starts the MLflow UI — click the printed link to browse them. Equally optional; equally explained in notebook 11.

In [ ]:
# optional: MLflow tracing (see notebook 11)
import obs

obs.enable_mlflow()
obs.mlflow_ui()            # prints the link where the traces appear

Boilerplate — same code as notebooks 03 and 04. Skim and run.

In [ ]:
# === boilerplate: aldervane micro-world — same code as notebook 03 ===
import json, math, re
from collections import Counter
from pathlib import Path

CORPUS = json.loads(Path("data/corpus.json").read_text())

STOPWORDS = {
    "a", "an", "and", "as", "at", "by", "did", "do", "does", "each", "every",
    "for", "from", "he", "her", "his", "in", "is", "it", "its", "of", "on",
    "or", "she", "that", "the", "this", "to", "was", "were", "what", "when",
    "where", "which", "who", "whom", "whose", "with",
}

def tokenize(text):
    words = re.findall(r"[a-z0-9]+", text.lower())
    # naive stemming: "captains" -> "captain", "records" -> "record"
    words = [w[:-1] if len(w) > 3 and w.endswith("s") and not w.endswith("ss") else w
             for w in words]
    return [w for w in words if w not in STOPWORDS]

DOC_TOKENS = {p["id"]: set(tokenize(p["title"] + " " + p["text"])) for p in CORPUS}
IDF = Counter()
for tokens in DOC_TOKENS.values():
    IDF.update(tokens)
IDF = {t: math.log(len(CORPUS) / df) for t, df in IDF.items()}

def search(query, k=2):
    """Return the k corpus passages that best match the query (keyword + IDF)."""
    q = set(tokenize(query))
    ranked = sorted(
        CORPUS,
        key=lambda p: (-sum(IDF.get(t, 0.0) for t in q & DOC_TOKENS[p["id"]]), p["id"]),
    )
    return ranked[:k]

def as_context(passages):
    return [f"{p['title']}: {p['text']}" for p in passages]

class HopQA(dspy.Module):
    """Two-hop retrieval QA over the Aldervane corpus."""

    def __init__(self, k=2):
        super().__init__()
        self.k = k
        self.hop1 = dspy.Predict("question -> search_query")
        self.hop2 = dspy.Predict("question, context: list[str] -> search_query")
        self.answer = dspy.ChainOfThought("question, context: list[str] -> answer")

    def forward(self, question):
        q1 = self.hop1(question=question).search_query
        first = search(q1, k=self.k)
        q2 = self.hop2(question=question, context=as_context(first)).search_query
        second = search(q2, k=self.k)
        passages = {p["id"]: p for p in first + second}      # merge + dedupe
        pred = self.answer(question=question, context=as_context(list(passages.values())))
        return dspy.Prediction(
            answer=pred.answer,
            reasoning=pred.reasoning,
            queries=[q1, q2],
            retrieved_ids=sorted(passages),
        )

In [ ]:
# === boilerplate: data + metric — same code as notebook 04 ===
QA = json.loads(Path("data/qa.json").read_text())

def to_examples(rows):
    return [dspy.Example(**row).with_inputs("question") for row in rows]

train, dev, test = (to_examples(QA[s]) for s in ("train", "dev", "test"))

def norm(s):
    s = re.sub(r"[^a-z0-9\s]", " ", s.lower())
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def em_metric(example, pred, trace=None):
    """1.0 if the predicted answer exactly matches the gold answer (after normalization)."""
    return norm(pred.answer) == norm(example.answer)

In [ ]:
evaluate = dspy.Evaluate(devset=dev, metric=em_metric, num_threads=8, display_progress=True)

baseline_score = evaluate(HopQA()).score
baseline_score

## Why rewrite instructions at all?

Demos teach by example; instructions teach by *description*. Each has an economics:

- **Demos** are concrete and reliable, but they spend tokens on every call, and they only cover situations resembling the examples.
- **Instructions** are compact (one paragraph amortized over every call), generalize past the examples, and can state *rules* — "answer with the shortest exact phrase", "the second query must name an entity from the first hop's results" — that no small set of demos can pin down.

Handwritten instructions are where prompt engineering lives. The question of this notebook: can the search loop from notebook 05 — generate candidates, score with the metric, keep the best — write them *for* us?

The lineage starts with **OPRO** (*Large Language Models as Optimizers*, Yang et al., 2023, [arXiv:2309.03409](https://arxiv.org/abs/2309.03409)): show an LLM the history of `(instruction, score)` pairs and ask it to propose an instruction that scores higher. An optimization loop where the *proposer* is itself an LLM. DSPy's `COPRO` applies that per-predictor:

| OPRO paper concept | Plain English | Where it lives in code |
|---|---|---|
| Meta-prompt with solution/score trajectory | The proposer LLM reads previous instructions and their measured scores | maintained internally by `COPRO` per predictor |
| Proposal step | "Write an instruction likely to score better" | `breadth` new candidates per round |
| Iterative refinement | Best candidates seed the next round | `depth` rounds |
| Objective evaluation | Each candidate instruction is actually scored | your `qa_metric` over the trainset slice |

In [ ]:
def qa_metric(example, pred, trace=None):
    """Normalized exact match; strict gate when bootstrapping (see notebook 05)."""
    correct = norm(pred.answer) == norm(example.answer)
    if trace is not None:
        return correct
    return float(correct)

*Cost note: COPRO evaluates `breadth` candidates × `depth` rounds × 3 predictors on the trainset slice — a few hundred LM calls at the settings below.*

One sharp edge, worth knowing before you run it: `COPRO.compile` **requires** the `eval_kwargs` argument (settings for its internal evaluations) — omitting it is a `TypeError`. It is the one optimizer in this course with that quirk.

In [ ]:
copro = dspy.COPRO(metric=qa_metric, breadth=4, depth=2)

copro_prog = copro.compile(
    HopQA(),
    trainset=train[:12],                                   # a slice keeps this affordable
    eval_kwargs=dict(num_threads=8, display_progress=False),
)

copro_score = evaluate(copro_prog).score
print(f"baseline {baseline_score} -> COPRO {copro_score}")

In [ ]:
for name, predictor in copro_prog.named_predictors():
    print(f"--- {name} ---")
    print(predictor.signature.instructions, "\n")

> **What you should see:** three rewritten instruction paragraphs — no longer the auto-generated `"Given the fields..."` but task-specific prose the proposer invented (often mentioning search queries, follow-up lookups, concise answers). The dev score may beat the baseline or roughly tie at this scale; the artifact to study is the *text*. You are reading machine-written prompt engineering, selected by measurement.

## MIPROv2: propose with your eyes open, search with a model

COPRO's proposer works nearly blind — it sees instructions and scores, nothing else. And it tunes instructions *only*, ignoring what notebook 05 proved about demos. **MIPROv2** (*Optimizing Instructions and Demonstrations for Multi-Stage Language Model Programs*, Opsahl-Ong et al., 2024, [arXiv:2406.11695](https://arxiv.org/abs/2406.11695)) upgrades both decisions:

```
 stage 1  BOOTSTRAP          run the notebook-05 machinery -> pools of candidate
          demo candidates    demo sets, per predictor
                                        |
 stage 2  GROUNDED           a proposer LM writes candidate instructions while reading:
          instruction        - a summary of your DATASET        (data_aware)
          proposal           - a summary of your PROGRAM's code (program_aware)
                             - sample execution traces
                             - a library of prompting tips      (tip_aware)
                                        |
 stage 3  BAYESIAN SEARCH    a surrogate model (TPE) proposes combinations
          over (instruction  (which instruction? which demo set? per predictor),
          x demo) configs    scores them on MINIBATCHES of the trainset,
                             occasionally re-checks leaders on the full set
```

The paper ↔ code map:

| MIPROv2 paper concept | Plain English | Where it lives in code |
|---|---|---|
| Multi-stage LM program | Several prompted modules composed in one pipeline | your `HopQA` with its 3 `named_predictors()` |
| Demonstration bootstrapping | Reuse metric-filtered teacher traces as candidate demos | `max_bootstrapped_demos=`, `max_labeled_demos=` |
| Grounded proposal | Proposer sees dataset summary, program summary, traces, tips — not just scores | `data_aware_proposer=`, `program_aware_proposer=`, `tip_aware_proposer=` on `compile` |
| Bayesian surrogate (TPE) search | Model which combos look promising instead of sampling blindly | the `compile` search loop; `num_candidates=`, budget via `auto="light"/"medium"/"heavy"` |
| Stochastic minibatch evaluation | Score candidates on small batches; full evals only for leaders | `minibatch=True` (default) |
| Separate proposer/task models | A strong model can write prompts that a cheap model will run | `prompt_model=` / `task_model=` (constructor) |

*Cost note: `auto="light"` on this task ≈ several hundred LM calls (bootstrapping + proposals + trials). The run below is the biggest of the course so far — a few minutes.*

In [ ]:
mipro = dspy.MIPROv2(
    metric=qa_metric,
    auto="light",                 # budget preset: "light" / "medium" / "heavy"
    max_bootstrapped_demos=4,
    max_labeled_demos=4,
)

mipro_prog = mipro.compile(
    HopQA(),
    trainset=train,
    valset=dev,
    program_aware_proposer=True,
    data_aware_proposer=True,
    tip_aware_proposer=True,
)

mipro_score = evaluate(mipro_prog).score
print(f"baseline {baseline_score} -> COPRO {copro_score} -> MIPROv2 {mipro_score}")

> **What you should see:** logs from all three stages (bootstrapping, proposal, then numbered trials with minibatch scores), and a final program that is typically **at or above** the notebook-05 result.
>
> | program | dev EM (16 q) | expect | notebook |
> |---|---|---|---|
> | `HopQA()` zero-shot | *yours* | ~25–55 | 04 |
> | + `BootstrapFewShot` | *yours* | baseline +10–25 | 05 |
> | + `BootstrapRS` | *yours* | ≥ bootstrap | 05 |
> | + `COPRO` | *yours* | baseline ±10 | 06 |
> | + `MIPROv2 (light)` | *yours* | ≥ bootstrap-RS, ties common | 06 |
>
> Honesty at toy scale: with 16 dev questions, MIPROv2 often *ties* the bootstrap family. Its advantages compound with harder tasks, bigger budgets (`auto="medium"`+), and bigger eval sets — what you are here to learn is the *workflow and the artifacts*, which transfer unchanged (the capstone re-runs this on real data).

In [ ]:
naive = HopQA()
for (name, tuned) in mipro_prog.named_predictors():
    original = dict(naive.named_predictors())[name]
    print(f"--- {name} ---")
    print("before:", original.signature.instructions)
    print("after: ", tuned.signature.instructions)
    print("demos: ", len(tuned.demos), "\n")

In [ ]:
from pathlib import Path
Path("artifacts").mkdir(exist_ok=True)
mipro_prog.save("artifacts/06_mipro.json")
print("saved — open it and read the instructions keys")

> **What you should see:** per-predictor before/after instructions — the "after" typically references the task's actual structure (multi-hop lookup, entity-focused queries, terse answers), because the proposer *read your program and your data* before writing. And the saved JSON is the same familiar format as notebook 05's — same two θ slots, different optimizer filling them.
>
> One connective thought: instruction optimization shines brightest when the task has an **unstated rubric** — label definitions, style guides, tie-breaking rules that live in the metric but not in the signature. Our metric's hidden rubric is "the shortest exact phrase". Check the `answer` predictor's new instructions: did MIPROv2 *discover* the rubric from the data and write it down? (It frequently does — that moment is the whole pitch for this family.)

## Recap

| Concept | One-liner |
|---|---|
| instructions vs demos | describe the rule vs. show examples; both are θ, different economics |
| `COPRO(breadth, depth)` | OPRO-style: propose → score → refine, per predictor; `compile(..., eval_kwargs=...)` is **required** |
| `MIPROv2(auto=...)` | bootstrap demos + grounded proposals + Bayesian (TPE) search over combos |
| grounded proposal | proposer reads dataset/program/traces/tips (`*_aware_proposer` flags) |
| minibatch evaluation | cheap noisy scores for many candidates, full evals for leaders |
| read the result | before/after `signature.instructions`; same `save()` JSON as ever |

## Exercises

1. Budget experiment (costs real money — optional): rerun with `auto="medium"`. Does the extra search buy points on *this* task? Why would the answer differ on a harder one?
2. Rerun MIPROv2 with `program_aware_proposer=False, data_aware_proposer=False`. Read the proposed instructions side by side with the grounded ones — what concretely went missing?
3. You vs. the machine: hand-write your best instructions for all three predictors (`sig.with_instructions(...)`, or edit a saved JSON and `load` it), evaluate, and compare against `mipro_prog`. Which of you encoded the answer-style rubric better?

**Next up:** every optimizer so far learns from a bare number. But your metric *knows more* — which passage was missed, which entity was confused. Notebook 07 feeds that knowledge back as **text**, and lets a reflection LM rewrite the program: GEPA.